# Method Comparison KDE Analysis (CertCF vs NN vs GS)

This notebook compares proximity distributions across methods on the same benchmark run,
with per-dataset KDE plots for `L1` and `L2` distances.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

NOTEBOOK_DIR = Path.cwd().resolve()
ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from notebooks.utils import (
    annotate_common_success_subset,
    attach_tabular_manifoldness_metrics,
    load_result,
    load_tabular_manifold_resources,
    matched_success_summary,
    method_comparison_summary,
    plot_conditioned_proximity_kdes,
    plot_proximity_kdes_by_dataset,
    prepare_benchmark_df,
    setup_notebook_style,
    summarize_manifoldness,
)

setup_notebook_style()

CONFIG = "benchmark_full_all_200q_alpha0.99"
RESULT_FILENAME = f"{CONFIG}.parquet"
CONFIG_PATH = ROOT / f"configs/benchmarks/{CONFIG}.yaml"
_candidates = [
    Path("results") / RESULT_FILENAME,
    Path("..") / "results" / RESULT_FILENAME,
]
RESULT_PATH = next((p.resolve() for p in _candidates if p.exists()), None)
if RESULT_PATH is None:
    looked = '\n'.join(str(p.resolve()) for p in _candidates)
    raise FileNotFoundError(f"Missing result file. Looked in:\n{looked}")

DATASET_ORDER = ["adult", "compas", "german_credit", "give_me_some_credit", "heloc", "lending_club", "wisconsin_breast_cancer"]
METHOD_ORDER = ["certcf_top3", "nn", "gs"]
METHOD_LABELS = {
    "certcf_top3": "CertCF (boundary-random, top-3)",
    "nn": "Nearest Neighbor",
    "gs": "Growing Spheres",
    "dice": "DiCE",
}
PALETTE = {
    "certcf_top3": "#1f77b4",
    "nn": "#ff7f0e",
    "gs": "#2ca02c",
    "dice": "#d62728",
}


def _method_label_fn(frame: pd.DataFrame) -> pd.Series:
    qk = pd.to_numeric(frame.get("meta__query_k_candidates", np.nan), errors="coerce")
    labels = frame["run_name"].astype(str).copy()
    certcf_mask = labels.str.startswith("certcf") & qk.notna()
    labels.loc[certcf_mask] = qk.loc[certcf_mask].astype(int).map(lambda value: f"certcf_top{value}")
    return labels


## Load And Prepare Results

In [ ]:
df = load_result(RESULT_PATH, method_label_fn=_method_label_fn)

comparison_df = prepare_benchmark_df(
    df,
    methods=METHOD_ORDER,
    dataset_order=DATASET_ORDER,
    method_order=METHOD_ORDER,
)
plot_df = prepare_benchmark_df(
    df,
    methods=METHOD_ORDER,
    dataset_order=DATASET_ORDER,
    method_order=METHOD_ORDER,
    success_only_rows=True,
)


## KDE Distribution Comparison By Dataset

The first plots compare method-wise distributions of proximity (`L1`, `L2`) for each dataset.

In [ ]:
figures = plot_proximity_kdes_by_dataset(
    plot_df,
    dataset_order=DATASET_ORDER,
    method_order=METHOD_ORDER,
    palette=PALETTE,
    method_labels=METHOD_LABELS,
)
for _, fig, _ in figures:
    plt.show()


## Proximity Result Table

This table summarizes proximity with both mean and median, alongside validity and runtime.

In [ ]:
proximity_summary = method_comparison_summary(
    comparison_df,
    order=METHOD_ORDER,
)
display(proximity_summary.round(3))


## Proximity Table On Common Successful Queries

This table restricts the analysis to the intersection of query points for which **all methods** succeeded within each dataset.

In [ ]:
common_counts_df, matched_summary = matched_success_summary(
    comparison_df,
    order=METHOD_ORDER,
)
display(common_counts_df)
display(matched_summary.round(3))


## KDE Conditioned On Common-Success Membership

For each dataset and method, these plots compare the proximity distribution on successful queries that belong to the common-success intersection against the successful queries outside that intersection.

In [ ]:
subset_kde_df = annotate_common_success_subset(
    plot_df,
    by=("dataset", "query_idx"),
    method_col="method_label",
)

SUBSET_ORDER = ["Common-success subset", "Outside common subset"]
SUBSET_PALETTE = {
    "Common-success subset": "#1f77b4",
    "Outside common subset": "#d62728",
}

figures = plot_conditioned_proximity_kdes(
    subset_kde_df,
    dataset_order=DATASET_ORDER,
    method_order=METHOD_ORDER,
    subset_col="subset_group",
    subset_order=SUBSET_ORDER,
    subset_palette=SUBSET_PALETTE,
    method_titles=METHOD_LABELS,
)
for _, fig, _ in figures:
    plt.show()


## Manifoldness Metrics

We start with three plausibility proxies computed on successful counterfactuals only:

1. `kNN-5 (all train)` — mean distance to the 5 nearest training points.
2. `kNN-5 (predicted target class)` — mean distance to the 5 nearest training points whose **predicted** class matches the counterfactual class.
3. `-LOF (all train)` — the negative of the Local Outlier Factor novelty score on the training set.

How to read them:

- Lower is better for `kNN-5 (all train)`: the counterfactual lies closer to the empirical data manifold.
- Lower is better for `kNN-5 (predicted target class)`: the counterfactual looks more like a realistic member of the class it flips into.
- Lower is better for `-LOF (all train)`: smaller values correspond to less outlier-like points, while larger values indicate stronger outlier behavior.

In practice, the second metric is often the most informative one for counterfactual realism, because it checks whether the CF lands near plausible examples of its **target** class rather than just near any training point.

In [ ]:
resources_by_dataset = load_tabular_manifold_resources(
    CONFIG_PATH,
    datasets=DATASET_ORDER,
    device="cpu",
)
manifold_df = attach_tabular_manifoldness_metrics(
    plot_df,
    resources_by_dataset=resources_by_dataset,
)
manifold_summary = summarize_manifoldness(
    manifold_df,
    order=METHOD_ORDER,
)
display(manifold_summary.round(3))


## Notes

- KDE curves are computed on successful counterfactuals only.
- To compare failure behavior, use the summary table (`valid_pct`) above.
- The matched-success table is stricter: it compares methods only on query points solved by all of them.
- The conditioned KDE plots show how each method's successful-query distribution changes inside versus outside that common subset.